# 06 Lab — Time Spreads: Calendars, Diagonals, and the PMCC

This lab builds each time-spread structure on the DEMO chain (spot 100) and shows why mixed-expiry
positions **must** be valued with `payoff.pnl_at` at the front-leg's expiry, not with the naive
intrinsic expiry diagram. You will:

1. Build an ATM calendar and draw its true P&L tent at front expiry.
2. Measure the calendar's vega sign with a vol-shifted scenario grid.
3. Build a bullish diagonal and a poor man's covered call and summarize them.

Run top-to-bottom. Nothing here touches the network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, greeks, viz, pricing, data

chain = data.load_sample_chain("DEMO")
SPOT = 100.0
chain.head()

## 1. The ATM calendar

Sell the 100 call at 21 DTE (mid 2.66), buy the 100 call at 45 DTE (mid 3.91). Same strike, same
kind, different expiries. Expiries are passed **in years**.

In [ ]:
cal = strategies.calendar_spread(
    "call", 100.0,
    front_expiry=21/365, front_premium=2.66,
    back_expiry=45/365, back_premium=3.91,
)
print(cal.describe())
print("net premium ($, + = debit):", cal.net_premium())

## 2. Why the intrinsic diagram lies

Plot the plain expiry payoff (all legs at intrinsic) **against** the true mark-to-model curve at the
front leg's expiry (`t_elapsed = 21/365`). Only the second curve is real: when the front expires the
45-DTE back call is still alive and still holds extrinsic value.

In [ ]:
spots = np.linspace(80, 120, 121)
naive = payoff.pnl_curve(cal, spots)                       # WRONG for mixed expiry
real = payoff.pnl_curve(cal, spots, t_elapsed=21/365, vol=0.26)  # correct
fig, ax = plt.subplots()
ax.plot(spots, naive, "--", label="naive intrinsic (misleading)")
ax.plot(spots, real, label="pnl_at front expiry (correct)")
ax.axhline(0, color="k", lw=0.8); ax.axvline(100, color="grey", lw=0.6)
ax.set_xlabel("spot"); ax.set_ylabel("P&L $"); ax.legend(); ax.set_title("Calendar: tent is real, V is fake")

The correct curve is the familiar **tent**, peaking at the strike (100). Confirm the peak
numerically at three spots.

In [ ]:
for s in (90, 100, 110):
    print(s, round(payoff.pnl_at(cal, s, t_elapsed=21/365, vol=0.26), 2))

## 3. The calendar is long vega

Use `analyzer.scenario_grid` with `vol_shift` to move IV up and down while holding spot at 100 and
stepping 10 days forward. A calendar should gain when vol **rises** (net long vega).

In [ ]:
grid = analyzer.scenario_grid(
    cal, spots=[100.0], days_forward=[10], vol=0.26,
    vol_shift=[-0.03, 0.0, +0.03],
)
grid[["spot", "days_forward", "vol", "pnl"]]

The highest `pnl` is at the highest `vol` — that is what "long vega" means. This is also
the warning: a broad vol crush that hits the back leg hurts a calendar.

## 4. Position greeks snapshot

`greeks.position_greeks` confirms the profile at entry: near delta-neutral, long theta (positive),
long vega (positive).

In [ ]:
g = greeks.position_greeks(cal, SPOT, vol=0.26)
print("delta", round(g.delta, 2), "theta", round(g.theta, 2), "vega", round(g.vega, 2))

## 5. Bullish call diagonal

Buy the 100 call at 45 DTE (3.91), sell the 105 call at 21 DTE (0.84). Different strikes add a
directional lean; the short leg subsidizes the debit.

In [ ]:
diag = strategies.diagonal_spread(
    "call", short=(105.0, 0.84), long=(100.0, 3.91),
    short_expiry=21/365, long_expiry=45/365,
)
print(diag.describe())
print("net debit $:", diag.net_premium())

Compare the diagonal's true front-expiry curve against the calendar to see the directional
skew.

In [ ]:
ax = viz.plot_compare([cal, diag])
ax.set_title("Calendar (symmetric) vs call diagonal (skewed up)")

## 6. Poor man's covered call

Buy a deep-ITM long-dated call (82.5 strike, 180 DTE) as synthetic stock, sell a near OTM call
(105, 21 DTE) against it. Price the long leg from the chain's IV with BSM.

In [ ]:
iv_82 = chain[(chain.kind=="call") & (chain.strike==82.5) & (chain.expiry_days==180)].iv.iloc[0]
long_prem = pricing.bsm_price("call", SPOT, 82.5, 180/365, iv_82)
pmcc = strategies.poor_mans_covered_call(
    long_call=(82.5, round(long_prem, 2)), short_call=(105.0, 0.84),
    long_expiry=180/365, short_expiry=21/365,
)
print("long call model price:", round(long_prem, 2))
analyzer.summarize(pmcc, SPOT, vol=iv_82)

Read `max_profit`, `max_loss`, and `net_premium` from the summary. The PMCC only works if
repeated short-call credits grind the debit basis down over many cycles — one cycle is rarely enough.

## Experiments

1. Move the calendar strike to 105 (sell 105@21 = 0.84, buy 105@45 = 1.85). How does the tent shift,
   and what directional view does an OTM calendar express?
2. In the vega grid, change `days_forward` to `[0, 10, 20]`. Watch the tent grow as front expiry
   nears — then note the gamma risk in the last days.
3. Widen the diagonal short strike to 110@21 (0.18). How much less subsidy, and how much more upside
   room did you buy?
4. In the PMCC, pick a long strike of 90 instead of 82.5. Does the shallower-ITM long leg's larger
   extrinsic hurt your theta? Compare `position_greeks` theta for both.
5. Re-run the calendar with `vol=0.20` everywhere. A lower vol regime shrinks the tent — why?